In [1]:
import pandas as pd
from os.path import join


def 성명_마스킹(value):
    if pd.isna(value):
        return value

    value = str(value)
    if len(value) <= 2:
        return value[:1] + "○" * (len(value) - 1)

    return value[:1] + "○" * (len(value) - 2) + value[-1]

def 주민번호_마스킹(value):
    if pd.isna(value):
        return value

    value = str(value)
    return value[:8] + "*" * (len(value) - 8)

In [2]:
# 자산정리 파일(여러 시트, 시트마다 다른 칼럼명)을 읽어서 이름과 주민번호를 마스킹. 유사 파일도 수정해서 사용가능
######################################################
path_dir = r"C:\Users\DATA\Documents\하이웍스 받은파일"
path_fn = "POOL8_데이터_260731_회계법인 발송"

성명칼럼들 = ["성명", "분납자성명인", "신청인명", "주채무자성명"]
주민번호칼럼들 = ["주민등록번호", "신청인주민번호", "주채무자주민번호"]
제외할시트 = ["요약"]
######################################################
path_excel = join(path_dir, f"{path_fn}.xlsx")
path_rst_excel = join(path_dir, f"{path_fn}_마스킹.xlsx")

In [3]:
excel_file = pd.ExcelFile(path_excel)

비식별칼럼들 = 성명칼럼들 + 주민번호칼럼들
저장된_시트수 = 0

with pd.ExcelFile(path_excel) as excel_file:
    with pd.ExcelWriter(path_rst_excel, engine="openpyxl") as writer:
        for sheet_name in excel_file.sheet_names:
            if sheet_name in 제외할시트:
                continue

            df = pd.read_excel(excel_file, sheet_name=sheet_name)

            for col in 성명칼럼들:
                if col in df.columns:
                    df[col] = df[col].apply(성명_마스킹)

            for col in 주민번호칼럼들:
                if col in df.columns:
                    df[col] = df[col].apply(주민번호_마스킹)

            저장할칼럼들 = [
                col for col in df.columns
                if col in 비식별칼럼들
            ]

            if 저장할칼럼들:
                df[저장할칼럼들].to_excel(
                    writer,
                    sheet_name=sheet_name,
                    index=False,
                )
                저장된_시트수 += 1

if 저장된_시트수 == 0:
    print("비식별 대상 칼럼이 있는 시트가 없습니다.")
else:
    print(f"저장 완료: {path_rst_excel}")
    

저장 완료: C:\Users\DATA\Documents\하이웍스 받은파일\POOL8_데이터_260731_회계법인 발송_마스킹.xlsx
